# Tutorial 3: Bayesian Inference with MCMC

With a trained emulator we can evaluate the likelihood at any parameter point
in milliseconds, making MCMC practical.

This notebook:
1. Defines the **prior** and **likelihood**
2. Runs **emcee** (ensemble sampler) — fast, easy to use
3. Diagnoses chain convergence with trace plots

> This notebook continues from Tutorial 2. Make sure you have run it first
> (or re-run the emulator training cells below).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hic_bayes import (
    fit_pca, Emulator, UniformPrior,
    mvn_loglike, build_covariance,
    run_emcee, thin_chain,
)
from hic_bayes.plots import plot_trace

# ---- Re-train emulator (copy of Tutorial 2 essentials) ----
X     = pd.read_csv('../data/toy/design_points.csv').values
Y     = pd.read_csv('../data/toy/simulation.csv').values
y_exp = pd.read_csv('../data/toy/experiment.csv').values.ravel()
y_err = pd.read_csv('../data/toy/experiment_error.csv').values.ravel()

pca = fit_pca(Y, n_components=5)
Z_train = pca.transform(Y)

param_ranges = np.column_stack([X.min(axis=0), X.max(axis=0)])
emulator = Emulator(n_restarts=4)
emulator.fit(X, Z_train, param_ranges, n_jobs=-1)
print('Emulator ready.')

In [ ]:
# Prior — independent uniform on each parameter
param_names = ['eta_s', 'zeta_s_max', 'tau_0', 'Qs_mu', 'T_switch']
prior = UniformPrior(
    mins=X.min(axis=0),
    maxs=X.max(axis=0),
)

# Project experimental data into PC space
y_exp_pc  = pca.transform(y_exp.reshape(1, -1)).ravel()
exp_var_pc = np.abs(pca.transform(y_err.reshape(1, -1))).ravel() ** 2

def log_prob(theta):
    lp = prior.log_prob(theta)
    if not np.isfinite(lp):
        return -np.inf
    y_pred_pc, y_std_pc = emulator.predict(theta.reshape(1, -1), return_std=True)
    cov = build_covariance(y_std_pc.ravel()**2, exp_var_pc)
    return lp + mvn_loglike(y_exp_pc - y_pred_pc.ravel(), cov)

# Quick sanity check
test_theta = prior.sample(1, seed=0).ravel()
print(f'log_prob at a random prior point: {log_prob(test_theta):.2f}')

In [ ]:
N_WALKERS = 100   # increase to 500+ for production runs
N_STEPS   = 500   # increase to 5000+ for production runs
n_params  = prior.n_params

# Initialise walkers as a small ball around the prior centre
p0 = prior.sample(N_WALKERS, seed=42)

print(f'Running emcee: {N_WALKERS} walkers × {N_STEPS} steps ...')
chain, log_prob_arr = run_emcee(
    log_prob, N_WALKERS, N_STEPS, n_params, p0,
    output_file='../results/chain_tutorial.hdf5',
    progress=True,
)
print(f'Chain shape: {chain.shape}  (walkers, steps, params)')

In [ ]:
param_labels = [
    r'$\eta/s$', r'$(\zeta/s)_{\rm max}$',
    r'$\tau_0$', r'$Q_s/\mu$', r'$T_{\rm switch}$'
]
BURNIN = N_STEPS // 2

fig = plot_trace(chain, param_labels, burnin=BURNIN)
plt.show()

# Mean acceptance fraction (should be 0.2 – 0.5 for a healthy chain)
# Recompute from log_prob changes as a proxy
print(f'\nSuggested burn-in: {BURNIN} steps')
print('If traces are not well-mixed, increase N_STEPS or N_WALKERS.')

In [ ]:
flat_chain = thin_chain(chain, burnin=BURNIN, thin=2)
print(f'Flat chain: {flat_chain.shape[0]} samples × {flat_chain.shape[1]} params')

# Quick marginal histograms
fig, axes = plt.subplots(1, n_params, figsize=(14, 3))
for i, (ax, label) in enumerate(zip(axes, param_labels)):
    ax.hist(flat_chain[:, i], bins=40, density=True)
    ax.set_xlabel(label)
    ax.set_ylabel('Posterior density')
plt.tight_layout()
plt.show()